# CXAS: process a folder

Segment every image in a folder (`.dcm`, `.png`, `.jpg`, subfolders included) and save the masks.

The example images come from the [NCI Imaging Data Commons](https://portal.imaging.datacommons.cancer.gov/) (CC BY): a PA + lateral pair of one patient from LIDC-IDRI and a portable AP film from COVID-19-NY-SBU. `demo_utils.get_example_images()` downloads them once into `example_data/` (needs `pip install idc-index`); sources and citations are in [`demo_utils.py`](demo_utils.py).

In [1]:
# !pip install cxas idc-index

In [2]:
import warnings; warnings.filterwarnings("ignore")
import transformers; transformers.utils.logging.disable_progress_bar()
import matplotlib.pyplot as plt; plt.rcParams["figure.dpi"] = 72
import torch
from demo_utils import get_example_images, display_image, overlay

DEVICE = "0" if torch.cuda.is_available() else "cpu"     # a GPU number, or "cpu"
paths = get_example_images()                              # downloads ~30 MB from the Imaging Data Commons once
paths

{'frontal_pa': 'example_data/frontal_pa.dcm',
 'lateral': 'example_data/lateral.dcm',
 'frontal_ap_portable': 'example_data/frontal_ap_portable.dcm',
 'frontal_pa_png': 'example_data/png/frontal_pa.png',
 'frontal_pa_jpg': 'example_data/jpg/frontal_pa.jpg',
 'dir': 'example_data'}

In [3]:
import os
from cxas import CXAS

model = CXAS(model_name="M2F_anatomy_v2", gpus=DEVICE)

model.process_folder(paths["dir"], "out/folder_npy", storage_type="npy", create=True)
for root, _, files in sorted(os.walk("out/folder_npy")):
    if files:
        print(root, sorted(files))

  0%|          | 0/5 [00:00<?, ?it/s]

 20%|██        | 1/5 [00:01<00:07,  1.86s/it]

 40%|████      | 2/5 [00:03<00:05,  1.74s/it]

 60%|██████    | 3/5 [00:06<00:04,  2.11s/it]

 80%|████████  | 4/5 [00:07<00:01,  1.92s/it]

100%|██████████| 5/5 [00:10<00:00,  2.17s/it]

100%|██████████| 5/5 [00:10<00:00,  2.06s/it]

out/folder_npy ['frontal_ap_portable.npy', 'frontal_pa.npy', 'lateral.npy']


One `.npy` file per image, shaped (159, height, width). The `json` storage type writes all images into one COCO file instead:

In [4]:
import json, glob

model.process_folder(paths["dir"], "out/folder_json", storage_type="json", create=True)
coco = json.load(open(glob.glob("out/folder_json/*.json")[0]))
print(len(coco["images"]), "images,", len(coco["annotations"]), "annotations,", len(coco["categories"]), "categories")

  0%|          | 0/5 [00:00<?, ?it/s]

 20%|██        | 1/5 [00:02<00:08,  2.24s/it]

 40%|████      | 2/5 [00:04<00:06,  2.18s/it]

 60%|██████    | 3/5 [00:07<00:04,  2.44s/it]

 80%|████████  | 4/5 [00:09<00:02,  2.33s/it]

100%|██████████| 5/5 [00:12<00:00,  2.82s/it]

100%|██████████| 5/5 [00:12<00:00,  2.59s/it]

5 images, 687 annotations, 159 categories


The same from the command line:

```bash
cxas_segment -i example_data/ -o out/folder_npy -ot npy -m M2F_anatomy_v2 -g cpu
```